# PART2 Coding Final Exam — Deep Learning 2026
**Duration:** 2 hours &nbsp;|&nbsp; **Total:** 100 points

---

## Instructions
- Fill in **all `TODO` sections**. Do **not** change function signatures or test cells.
- Each part has: **(1) your implementation → (2) auto-graded tests → (3) end-to-end demo**
- Run the demo cell to verify your code works before submitting.
- Standard libraries (`math`, `copy`, `random`) are available. No external downloads needed.
- Every part is based on code you already wrote/saw in the assignments.

| Part | Topic | From | Points |
|------|-------|------|--------|
| 1 | Segmentation mean-IoU metric | A2 — Segmentation | 25 |
| 2 | DINO — self-distillation loss + teacher EMA | A3 — SSL | 40 |
| 3 | GraphSAGE (mean-pool aggregation) | A5 — GNN | 35 |
| **Total** | | | **100** |

In [1]:
# Install required packages (safe to re-run if already installed)
%pip install torch torchvision numpy pandas

import os
os.environ['http_proxy']  = 'http://192.41.170.23:3128'
os.environ['https_proxy'] = 'http://192.41.170.23:3128'

Note: you may need to restart the kernel to use updated packages.


c:\Users\svrat\Documents\AIT\Inter-sem\RTML\.venv\Scripts\python.exe: No module named pip


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
import random
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


---
## Part 1 — Segmentation mean-IoU (25 points)

In **A2 (Image Segmentation)** you trained a U-Net and evaluated it with the
mean **IoU (Intersection-over-Union)** metric — the same `compute_iou` you used there.

For each class `c`, IoU compares the predicted mask against the ground truth:
```
IoU_c = |pred==c ∩ target==c| / |pred==c ∪ target==c|
```
The metric returns the **mean IoU over classes that actually appear** (classes with
empty union are skipped). `pred` are raw logits `(N, C, H, W)` → take `argmax` over the
channel dim to get predicted labels `(N, H, W)`; `target` holds class indices `(N, H, W)`.

In [28]:
def compute_iou(pred, target, n_classes=3):
    """
    Mean Intersection-over-Union for semantic segmentation  (from A2 — Segmentation).

    Args:
        pred    : (N, C, H, W) raw logits / probabilities
        target  : (N, H, W)    ground-truth class indices in [0, n_classes)
        n_classes: number of classes
    Returns:
        mean IoU (float) over classes whose union > 0; 0.0 if no class appears

    Steps:
        1. pred_labels = argmax over the channel dim (dim=1)  → (N, H, W)
        2. for each class c in range(n_classes):
             inter = count of pixels where (pred_labels==c) AND (target==c)
             union = count of pixels where (pred_labels==c) OR  (target==c)
             if union > 0: append (inter / union) to a list
        3. return mean of the list (np.mean), or 0.0 if the list is empty
    """
    # TODO 1a: pred_labels = argmax over channel dim
    pred_labels = pred.argmax(dim=1)  # (N, H, W)
    ious = []
    # TODO 1b: loop over classes, compute intersection & union, append IoU when union > 0
    for cls in range(n_classes):
        inter = ((pred_labels==cls) & (target==cls)).sum().float()
        union = ((pred_labels==cls) | (target==cls)).sum().float()
        if union > 0:
            ious.append((inter / union).item())

    # TODO 1c: return mean IoU (or 0.0 if empty)
    return np.mean(ious) if ious else 0.0

## What to implement: `compute_iou`

This is the **exact metric from A2 — Segmentation**. Fill in the three TODOs.

| Test | Points | What it checks |
|------|--------|----------------|
| T1 | 5 | Perfect prediction (pred argmax == target) → mean IoU = 1.0 |
| T2 | 5 | Correct value on a hand-made case with a known IoU |
| T3 | 10 | Classes absent from both pred & target (union = 0) are skipped, not counted as 0 |
| T4 | 5 | Works with a batch (N > 1); returns a single float |

**Tips**
- `pred.argmax(dim=1)` collapses the channel dim → predicted label map.
- `((a == c) & (b == c)).sum()` counts intersection pixels; use `|` for union.
- Guard with `if union > 0` so empty classes don't pollute the average.

In [4]:
# ── Part 1 Tests (25 points) ──────────────────────────────────────────────
_score_p1 = 0
torch.manual_seed(42)

In [29]:
# [T1-5pts] perfect prediction → IoU = 1.0
try:
    torch.manual_seed(0)
    _target = torch.tensor([[[0, 1], [2, 0]]])          # (1, 2, 2) class indices
    # build one-hot logits that argmax exactly to _target
    _pred = torch.zeros(1, 3, 2, 2)
    for _c in range(3):
        _pred[0, _c] = (_target[0] == _c).float() * 10.0
    _iou = compute_iou(_pred, _target, n_classes=3)
    assert abs(_iou - 1.0) < 1e-6, f"perfect pred should give 1.0, got {_iou}"
    _score_p1 += 5
    print("PASS T1 (5/5)  — perfect prediction → IoU 1.0")
except Exception as e:
    print(f"FAIL T1 (0/5)  — {e}")

PASS T1 (5/5)  — perfect prediction → IoU 1.0


In [30]:
# [T2-5pts] correct value on a known hand-made case
try:
    # target: 2 pixels class0, 2 pixels class1
    _target = torch.tensor([[[0, 0], [1, 1]]])          # (1, 2, 2)
    # pred labels: [0,0,0,1] → class0 inter=2 union=3 (IoU 2/3); class1 inter=1 union=2 (IoU 1/2)
    _lab  = torch.tensor([[[0, 0], [0, 1]]])
    _pred = torch.zeros(1, 2, 2, 2)
    for _c in range(2):
        _pred[0, _c] = (_lab[0] == _c).float() * 10.0
    _iou = compute_iou(_pred, _target, n_classes=2)
    _exp = ( (2/3) + (1/2) ) / 2
    assert abs(_iou - _exp) < 1e-6, f"expected {_exp:.4f}, got {_iou:.4f}"
    _score_p1 += 5
    print("PASS T2 (5/5)  — correct mean IoU value")
except Exception as e:
    print(f"FAIL T2 (0/5)  — {e}")

PASS T2 (5/5)  — correct mean IoU value


In [31]:
# [T3-10pts] classes absent from both pred & target (union=0) are skipped
try:
    # only classes 0 and 1 ever appear, but n_classes=3 → class 2 has union 0 and must be skipped
    _target = torch.tensor([[[0, 1], [1, 0]]])
    _lab    = torch.tensor([[[0, 1], [1, 0]]])           # perfect on classes 0,1
    _pred = torch.zeros(1, 3, 2, 2)
    for _c in range(3):
        _pred[0, _c] = (_lab[0] == _c).float() * 10.0
    _iou = compute_iou(_pred, _target, n_classes=3)
    # if class 2 were counted as 0.0 the mean would be (1+1+0)/3 = 0.667; skipping → 1.0
    assert abs(_iou - 1.0) < 1e-6, f"absent class must be skipped, got {_iou:.4f}"
    _score_p1 += 10
    print("PASS T3 (10/10) — empty-union classes skipped")
except Exception as e:
    print(f"FAIL T3 (0/10) — {e}")

PASS T3 (10/10) — empty-union classes skipped


In [32]:
# [T4-5pts] batch dimension handled, returns a single float
try:
    torch.manual_seed(1)
    _target = torch.randint(0, 3, (4, 8, 8))            # (N=4, H, W)
    _pred   = torch.randn(4, 3, 8, 8)
    _iou = compute_iou(_pred, _target, n_classes=3)
    assert isinstance(_iou, float), f"should return a float, got {type(_iou)}"
    assert 0.0 <= _iou <= 1.0, f"IoU out of range: {_iou}"
    _score_p1 += 5
    print("PASS T4 (5/5)  — batch handled, scalar in [0,1]")
except Exception as e:
    print(f"FAIL T4 (0/5)  — {e}")

PASS T4 (5/5)  — batch handled, scalar in [0,1]


In [33]:
print(f"\nPart 1 Score: {_score_p1} / 25")


Part 1 Score: 25 / 25


### Part 1 Demo — mean-IoU on a synthetic mask (not graded)

Run this to see `compute_iou` behave like it did in A2 — a slightly-corrupted
prediction scores below a perfect one. No dataset download needed.

In [34]:
# ── Part 1 Demo (not graded) ──────────────────────────────────────────────
torch.manual_seed(0)

# synthetic 3-class ground-truth mask (1, 16, 16)
_gt = torch.randint(0, 3, (1, 16, 16))

# perfect prediction
_perfect = torch.zeros(1, 3, 16, 16)
for _c in range(3):
    _perfect[0, _c] = (_gt[0] == _c).float() * 10.0

# corrupt 30% of pixels
_corrupt_lab = _gt.clone()
_mask = torch.rand(1, 16, 16) < 0.3
_corrupt_lab[_mask] = torch.randint(0, 3, (int(_mask.sum()),))
_corrupt = torch.zeros(1, 3, 16, 16)
for _c in range(3):
    _corrupt[0, _c] = (_corrupt_lab[0] == _c).float() * 10.0

print(f"Perfect prediction  mean-IoU: {compute_iou(_perfect, _gt):.4f}")
print(f"30%-corrupted pred  mean-IoU: {compute_iou(_corrupt, _gt):.4f}")
print("\n→ Corrupting pixels lowers IoU — the metric tracks segmentation quality")

Perfect prediction  mean-IoU: 1.0000
30%-corrupted pred  mean-IoU: 0.6152

→ Corrupting pixels lowers IoU — the metric tracks segmentation quality


---
## Part 2 — DINO: Self-Distillation (40 points)

In **A3 (Self-Supervised Learning)** you studied **DINO** — a student network learns
by matching the output of a **teacher** that is an EMA (exponential moving average) of
the student. You will re-implement its two core pieces:

### 1. Teacher EMA update  (from the A3 DINO training loop)
```
θ_teacher = m · θ_teacher + (1 - m) · θ_student      (no gradient)
```

### 2. `DINOLoss`  (the exact class from A3)
```
student output → log_softmax( s / student_temp )
teacher output → softmax( (t - center) / teacher_temp ).detach()      ← sharpened + centered
loss = mean over crop pairs (i≠j) of  -(teacher_prob · student_log_prob).sum()
center ← center_momentum · center + (1 - center_momentum) · mean(teacher_out)
```
Centering + sharpening (low teacher temp) is what stops DINO from collapsing.

In [96]:
def update_teacher(student_net, teacher_net, m):
    """
    EMA update of the teacher parameters  (from the A3 DINO training loop).
        θ_teacher = m * θ_teacher + (1 - m) * θ_student
    Use torch.no_grad(); update .data in place. Do NOT change the student.
    """
    # TODO 1: EMA update over zipped student/teacher parameters
    with torch.no_grad():
        for s_p, t_p in zip(student_net.parameters(), teacher_net.parameters()):
            t_p.data = m * t_p.data + (1 - m) * s_p.data
        
            
            
            
            


class DINOLoss(nn.Module):
    """DINO self-distillation loss with centering + sharpening  (from A3)."""
    def __init__(self, out_dim=256, teacher_temp=0.04, student_temp=0.1, center_momentum=0.9):
        super().__init__()
        self.student_temp    = student_temp
        self.teacher_temp    = teacher_temp
        self.center_momentum = center_momentum
        self.register_buffer('center', torch.zeros(1, out_dim))

    def forward(self, student_out, teacher_out):
        """
        Args:
            student_out : list of (N, out_dim) tensors — all crops
            teacher_out : list of (N, out_dim) tensors — global crops only
        Returns:
            scalar loss

        Steps:
            1. s_probs = [log_softmax(s / student_temp, dim=-1) for s in student_out]
            2. t_probs = [softmax((t - center) / teacher_temp, dim=-1).detach() for t in teacher_out]
            3. for every (teacher i, student j) with i != j:
                 add  -(t_prob * s_log_prob).sum(dim=-1).mean()
               average over the number of pairs
            4. call self.update_center(mean of stacked teacher_out)
            5. return the averaged loss
        """
        #TODO 2a: sharpen student (log_softmax) and teacher (softmax, centered, detached)
        s_probs = [F.log_softmax(s / self.student_temp, dim=-1) for s in student_out]
        t_probs = [F.softmax((t - self.center) / self.teacher_temp, dim=-1).detach()
                   for t in teacher_out]

        #TODO 2b: cross-view cross-entropy over all pairs (i != j), averaged
        total_loss = 0.0
        n_loss_terms = 0
        for t_idx, t_prob in enumerate(t_probs):
            for s_idx, s_log_prob in enumerate(s_probs):
                # skip same view: student global crop i vs teacher global crop i
                if s_idx == t_idx:
                    continue
                loss = -(t_prob * s_log_prob).sum(dim=-1).mean()
                total_loss += loss
                n_loss_terms += 1
                
        total_loss /= n_loss_terms
    
        # TODO 2c: update the center, then return the averaged loss
        self.update_center(torch.stack(teacher_out).mean(dim=0))
        return total_loss 

    @torch.no_grad()
    def update_center(self, teacher_mean):
        """center = center_momentum * center + (1 - center_momentum) * teacher_mean"""
        # TODO 3: EMA update of the center buffer (in place)
        self.center = self.center * self.center_momentum + (1 - self.center_momentum) * teacher_mean

## What to implement: `update_teacher` + `DINOLoss`

Both come straight from **A3 — DINO**. Fill in the TODOs.

| Test | Points | What it checks |
|------|--------|----------------|
| T5 | 10 | `update_teacher` EMA: teacher moves toward student, student untouched |
| T6 | 10 | `DINOLoss.forward` returns a finite scalar ≥ 0 on random crops |
| T7 | 5 | `update_center` EMA: center buffer updates in place by the right amount |
| T8 | 15 | Matching student/teacher gives lower loss than mismatched; center tracks teacher mean |

**Tips**
- EMA: `t_p.data = m * t_p.data + (1 - m) * s_p.data` inside `torch.no_grad()`.
- Teacher target is **detached** and **sharpened** (small `teacher_temp`) after subtracting `center`.
- Skip the pair where the student crop index equals the teacher crop index (`i == j`).

In [97]:
# ── Part 2 Tests (40 points) ──────────────────────────────────────────────
_score_p2 = 0
torch.manual_seed(42)

In [98]:
# [T5-10pts] teacher EMA update
try:
    _sn = nn.Linear(4, 4, bias=False); _tn = nn.Linear(4, 4, bias=False)
    nn.init.constant_(_sn.weight, 1.0); nn.init.constant_(_tn.weight, 0.0)
    update_teacher(_sn, _tn, m=0.9)
    assert abs(_tn.weight[0, 0].item() - 0.1) < 1e-6, f"teacher should be 0.1, got {_tn.weight[0,0].item()}"
    assert abs(_sn.weight[0, 0].item() - 1.0) < 1e-6, "student must be unchanged"
    _score_p2 += 10
    print("PASS T5 (10/10) — teacher EMA update correct")
except Exception as e:
    print(f"FAIL T5 (0/10) — {e}")

PASS T5 (10/10) — teacher EMA update correct


In [100]:
# [T6-10pts] DINOLoss.forward returns a finite scalar >= 0
try:
    torch.manual_seed(0)
    _dl = DINOLoss(out_dim=16)
    _student = [torch.randn(8, 16), torch.randn(8, 16), torch.randn(8, 16)]  # 3 crops
    _teacher = [torch.randn(8, 16), torch.randn(8, 16)]                       # 2 global crops
    _loss = _dl(_student, _teacher)
    assert _loss.shape == () and not torch.isnan(_loss), f"bad loss {_loss}"
    assert _loss.item() >= 0, f"cross-entropy loss should be >= 0, got {_loss.item()}"
    _score_p2 += 10
    print("PASS T6 (10/10) — DINOLoss forward valid")
except Exception as e:
    print(f"FAIL T6 (0/10) — {e}")

PASS T6 (10/10) — DINOLoss forward valid


In [101]:
# [T7-5pts] update_center EMA on the center buffer
try:
    _dl = DINOLoss(out_dim=8, center_momentum=0.9)
    assert torch.allclose(_dl.center, torch.zeros(1, 8)), "center should start at zero"
    _tmean = torch.ones(1, 8) * 2.0
    _dl.update_center(_tmean)
    # center = 0.9*0 + 0.1*2.0 = 0.2
    assert torch.allclose(_dl.center, torch.full((1, 8), 0.2), atol=1e-6), f"center wrong: {_dl.center}"
    _score_p2 += 5
    print("PASS T7 (5/5)   — center EMA update correct")
except Exception as e:
    print(f"FAIL T7 (0/5)   — {e}")

PASS T7 (5/5)   — center EMA update correct


In [102]:
# [T8-15pts] matched student/teacher → lower loss; center tracks teacher mean
try:
    torch.manual_seed(1)
    _feat = torch.randn(8, 16)
    # matched: student and teacher see the same features on two views
    _dl_m = DINOLoss(out_dim=16)
    _loss_match = _dl_m([_feat, _feat], [_feat, _feat])
    # mismatched: teacher sees unrelated features
    _dl_r = DINOLoss(out_dim=16)
    _loss_rand = _dl_r([_feat, _feat], [torch.randn(8, 16), torch.randn(8, 16)])
    assert _loss_match.item() < _loss_rand.item(), \
        f"matched loss {_loss_match.item():.3f} should be < mismatched {_loss_rand.item():.3f}"
    # center should have moved away from zero toward the teacher mean
    assert _dl_m.center.abs().sum().item() > 0, "center did not update during forward"
    _score_p2 += 15
    print("PASS T8 (15/15) — matched < mismatched; center updated")
except Exception as e:
    print(f"FAIL T8 (0/15) — {e}")

PASS T8 (15/15) — matched < mismatched; center updated


In [103]:
print(f"\nPart 2 Score: {_score_p2} / 40")


Part 2 Score: 50 / 40


### Part 2 Demo — DINO self-distillation on random features (not graded)

A tiny student/teacher MLP pair trained on random vectors. The loss should
drop as the student learns to match the (EMA) teacher. No download needed.

In [104]:
# ── Part 2 Demo (not graded) ──────────────────────────────────────────────
torch.manual_seed(0)
OUT_DIM = 32

# A student network distills the output distribution of a fixed (frozen) teacher.
# This isolates the DINOLoss behaviour: the loss should steadily decrease as the
# student learns to reproduce the teacher's sharpened targets. No download needed.
_student = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, OUT_DIM))
_teacher = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, OUT_DIM))
for _p in _teacher.parameters():
    _p.requires_grad = False                                 # frozen target teacher

_dino_loss = DINOLoss(out_dim=OUT_DIM)
_opt = torch.optim.Adam(_student.parameters(), lr=3e-3)

torch.manual_seed(1)
_x1, _x2 = torch.randn(32, 64), torch.randn(32, 64)          # two views (fixed)

print("DINO self-distillation (student matches a fixed teacher):")
for _step in range(50):
    _s_out = [_student(_x1), _student(_x2)]
    with torch.no_grad():
        _t_out = [_teacher(_x1), _teacher(_x2)]
    _loss = _dino_loss(_s_out, _t_out)
    _opt.zero_grad(); _loss.backward(); _opt.step()
    if (_step + 1) % 10 == 0:
        print(f"  step {_step+1:2d} | loss: {_loss.item():.4f}")

print("\n→ Loss decreases as the student reproduces the teacher's sharpened output")

DINO self-distillation (student matches a fixed teacher):
  step 10 | loss: 2.7963
  step 20 | loss: 2.2081
  step 30 | loss: 1.6963
  step 40 | loss: 1.4913
  step 50 | loss: 1.4070

→ Loss decreases as the student reproduces the teacher's sharpened output


---
## Part 3 — GraphSAGE with Mean-Pool Aggregation (35 points)

In **A5 (Graph Neural Networks)** you implemented **GraphSAGE**, which builds each node's
new embedding from **itself** and the **mean of its sampled neighbors**:
```
h_N(v) = mean{ h_u : u ∈ sampled_N(v) }          ← neighbor aggregation
h_v'   = ReLU( W · [ h_v ‖ h_N(v) ] )            ← concat self + neighborhood, then project
```
Neighbors are sampled with `random.choices(nbrs, k=min(k, len(nbrs)))`. If a node has **no
neighbors**, it aggregates **itself** (`agg[v] = H[v]`). This is the exact `GraphSAGELayer`
from A5 — you'll re-implement it plus the 2-layer `GraphSAGE` model.

In [105]:
class GraphSAGELayer(nn.Module):
    """GraphSAGE layer: h_v = ReLU( W · [h_v || mean(sampled_neighbors)] )   (from A5)."""
    def __init__(self, in_features, out_features):
        super().__init__()
        # TODO 1: W = Linear(in_features * 2, out_features)   (concat self + neighbor agg)
        self.W = nn.Linear(in_features * 2, out_features)

    def forward(self, H, adj_list, k=10):
        """
        Args:
            H        : (N, in_features) node features
            adj_list : list of lists — adj_list[v] = neighbor indices of node v
            k        : max neighbors to sample
        Returns:
            (N, out_features)

        Per node v:
            - no neighbors     → agg[v] = H[v]                       (aggregate self)
            - has neighbors    → sampled = random.choices(nbrs, k=min(k, len(nbrs)))
                                 agg[v]  = mean of H[sampled]  (dim=0)
        Finally: ReLU( W( cat([H, agg], dim=-1) ) )
        """
        N   = H.size(0)
        agg = torch.zeros(N, H.size(1), device=H.device)

        # TODO 2: fill agg[v] for every node
        for v in range(N):
            nbrs = adj_list[v]
            if len(nbrs) == 0:
                agg[v] = H[v]  # aggregate self
            else:
                sampled = random.choices(nbrs, k=min(k, len(nbrs)))
                agg[v] = H[sampled].mean(dim=0)

        # TODO 3: concat [H || agg], linear, ReLU
        return F.relu(self.W(torch.cat([H, agg], dim=-1)))


class GraphSAGE(nn.Module):
    """2-layer GraphSAGE for node classification   (from A5)."""
    def __init__(self, in_features, hidden_dim, n_classes, k=10, dropout=0.5):
        super().__init__()
        # TODO 4: layer1 (in→hidden), layer2 (hidden→n_classes), dropout, store k
        self.layer1  = GraphSAGELayer(in_features, hidden_dim)
        self.layer2  = GraphSAGELayer(hidden_dim, n_classes)
        self.dropout = nn.Dropout(dropout)
        self.k       = k

    def forward(self, X, adj_list):
        """Returns (logits, hidden).  layer1 → dropout → layer2."""
        # TODO 5: layer1 → dropout → layer2; return (logits, hidden_after_layer1)
        h = self.layer1(X, adj_list, self.k)
        h = self.dropout(h)
        out = self.layer2(h, adj_list, self.k)
        return out, h

## What to implement: `GraphSAGELayer` + `GraphSAGE`

This is the **mean-pool GraphSAGE from A5**. Fill in the TODOs.

| Test | Points | What it checks |
|------|--------|----------------|
| T9 | 10 | Output shape + mean-pool math on a single-neighbor node (deterministic) |
| T10 | 10 | A node with no neighbors aggregates itself (`agg[v] = H[v]`) |
| T11 | 15 | Full 2-layer `GraphSAGE.forward` returns `(logits, hidden)` with the right shape |

**Tips**
- `self.W = nn.Linear(in_features * 2, out_features)` — the input is the concat `[h_v ‖ agg]`.
- Sample with `random.choices(nbrs, k=min(k, len(nbrs)))`, then `.mean(dim=0)`.
- Empty neighborhood → `agg[v] = H[v]` (aggregate the node itself).

In [106]:
# ── Part 3 Tests (35 points) ──────────────────────────────────────────────
_score_p3 = 0
torch.manual_seed(42)
random.seed(42)

In [107]:
# [T9-10pts] output shape + mean-pool math on a single-neighbor node
try:
    torch.manual_seed(0)
    random.seed(0)
    _l = GraphSAGELayer(8, 16)
    _H = torch.randn(5, 8)
    # node 3 has exactly one neighbor [1] → random.choices([1]) == [1] always (deterministic)
    _adj = [[1, 2], [0, 3], [0, 4], [1], [2]]
    _out = _l(_H, _adj)
    assert _out.shape == (5, 16), f"shape {_out.shape}"
    with torch.no_grad():
        _agg3    = _H[1]                                    # single neighbor → mean is itself
        _exp_out = F.relu(_l.W(torch.cat([_H[3], _agg3])))  # ReLU(W([h_v || agg]))
    assert torch.allclose(_out[3], _exp_out, atol=1e-4), "mean-pool math mismatch for node 3"
    _score_p3 += 10
    print("PASS T9  (10/10) — shape + mean aggregation correct")
except Exception as e:
    print(f"FAIL T9  (0/10) — {e}")

PASS T9  (10/10) — shape + mean aggregation correct


In [108]:
# [T10-10pts] node with no neighbors aggregates itself
try:
    torch.manual_seed(1)
    random.seed(1)
    _l = GraphSAGELayer(4, 8)
    _H = torch.randn(3, 4)
    _adj = [[], [2], [1]]                                   # node 0 is isolated
    _out = _l(_H, _adj)
    with torch.no_grad():
        _exp0 = F.relu(_l.W(torch.cat([_H[0], _H[0]])))     # agg = self → [h_0 || h_0]
    assert torch.allclose(_out[0], _exp0, atol=1e-4), "isolated node should aggregate itself"
    _score_p3 += 10
    print("PASS T10 (10/10) — isolated node aggregates self")
except Exception as e:
    print(f"FAIL T10 (0/10) — {e}")

PASS T10 (10/10) — isolated node aggregates self


In [109]:
# [T11-15pts] full 2-layer GraphSAGE forward
try:
    torch.manual_seed(2)
    random.seed(2)
    _gm  = GraphSAGE(16, 32, 7, k=5)
    _X   = torch.randn(10, 16)
    _adj = [[1,2,3],[0,4],[0,5,6],[0,7],[1,8],[2,9],[2,3],[3],[4],[5]]
    _out = _gm(_X, _adj)
    assert isinstance(_out, tuple) and len(_out) == 2, "forward must return (logits, hidden)"
    _logits, _hidden = _out
    assert _logits.shape == (10, 7), f"logits shape {_logits.shape}"
    assert not torch.isnan(_logits).any(), "logits contain NaN"
    _score_p3 += 15
    print("PASS T11 (15/15) — GraphSAGE full forward")
except Exception as e:
    print(f"FAIL T11 (0/15) — {e}")

PASS T11 (15/15) — GraphSAGE full forward


In [110]:
print(f"\nPart 3 Score: {_score_p3} / 35")


Part 3 Score: 35 / 35


### Part 3 Demo — node classification on a tiny hand-built graph (not graded)

A small random graph trained for a few steps. The loss should drop as the
2-layer GraphSAGE learns. No download needed.

In [111]:
# ── Part 3 Demo (not graded) ──────────────────────────────────────────────
torch.manual_seed(0)
random.seed(0)

# tiny random graph: 20 nodes, 3 classes
_N, _F, _C = 20, 12, 3
_X = torch.randn(_N, _F)
_Y = torch.randint(0, _C, (_N,))
# random undirected edges
_adj = [[] for _ in range(_N)]
for _ in range(40):
    _a, _b = random.randrange(_N), random.randrange(_N)
    if _a != _b:
        _adj[_a].append(_b); _adj[_b].append(_a)

_model = GraphSAGE(_F, 32, _C, k=5)
_opt   = torch.optim.Adam(_model.parameters(), lr=0.01)

print("GraphSAGE node classification (30 steps, toy graph):")
for _step in range(30):
    _model.train()
    _logits, _ = _model(_X, _adj)
    _loss = F.cross_entropy(_logits, _Y)
    _opt.zero_grad(); _loss.backward(); _opt.step()
    if (_step + 1) % 10 == 0:
        _acc = (_logits.argmax(1) == _Y).float().mean().item()
        print(f"  step {_step+1:2d} | loss: {_loss.item():.4f} | train acc: {_acc:.2f}")

print("\n→ Loss drops as GraphSAGE learns to classify nodes from neighborhood structure")

GraphSAGE node classification (30 steps, toy graph):
  step 10 | loss: 1.0059 | train acc: 0.40
  step 20 | loss: 0.8034 | train acc: 0.55
  step 30 | loss: 0.6691 | train acc: 0.55

→ Loss drops as GraphSAGE learns to classify nodes from neighborhood structure


---
## Final Score

In [112]:
print("=" * 48)
print(f"  Part 1 — Segmentation mean-IoU : {_score_p1:>3} / 25")
print(f"  Part 2 — DINO self-distillation: {_score_p2:>3} / 40")
print(f"  Part 3 — GraphSAGE mean-pool   : {_score_p3:>3} / 35")
print("=" * 48)
print(f"  TOTAL                          : {_score_p1+_score_p2+_score_p3:>3} / 100")
print("=" * 48)

  Part 1 — Segmentation mean-IoU :  25 / 25
  Part 2 — DINO self-distillation:  50 / 40
  Part 3 — GraphSAGE mean-pool   :  35 / 35
  TOTAL                          : 110 / 100


In [113]:
print("Author: Vibolrottana Seng")
print("ID: st126425")
print("P2_Coding_RTML_Final")

Author: Vibolrottana Seng
ID: st126425
P2_Coding_RTML_Final
